<a href="https://colab.research.google.com/github/dominiksakic/NETworkingMay/blob/main/26_transformers.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Transformer weights/brains
- Embedding layers: Words converted into dense vectors.
- Attention Layers: Query, Key and Value Matrices.
- Feedforward layers: like in fully connected NN.
- Layer Norm
- Final output

# Intuition about - attention
- focused part of data, which is the result of an attention filter applied to the original.
- then focus on another part of the data, via another attention filter.
- Each focused result gets concated together and then projected back into the original dimension to avoid increasing the model size.

# What is so special?
- Linear layer: each output depends on each input. (Too much computation for large input and each output)
- Multi - head attention layer: every output depends on every input. But with less computation and weights.

# How does it do it?
- Linear layers: creates each output by taking the total input and different weights for each output. (Weighted sums)

- Attention layers
 - 1. store less amount weights, but can produce the weights in the process.*
 - 2. reduces the computation needed:
  - First a row wise computation and then a column wise computation. (Smilair to a Seperable Convolution)

 * *Dont stores  giants weights directly, they store a smaller amount of weights. The smaller amount of weights get created dynamically with the input to calculate the output/weighted sums.


 # Side notes:
 - Convolution are based on a small spacial area of the input.
 - Convolution share the exact weights with the other outputs.
 - CNNS: Local focus, shared weights across space.

 - Recurrent NN are based on a single point in of the input with a limited aggregation of the past.
 - The weights are shared with every other point in the sequence.
 - RNNS: Process input step-by-step, with shared weights overtime.

 - Transformers, each output still depends on the entire input and it still gets it unique weights.
 - Dynamic weight creation and the decomposition of the computation -> makes it so effective.
 - Full input context at every layer; dynamically computed weights allow flexible, parallel processing.





In [1]:
import numpy as np
"""
1.
 the cat sat

2. Token Vectors
[
  [0.1, 0.3, 0.5],  # embedding of word1
  [0.7, 0.2, 0.9],  # embedding of word2
  [0.0, 0.4, 0.2]   # embedding of word3
]

3. Self attention

scores = [
  dot(word1, word1),
  dot(word1, word2),
  dot(word1, word3)
]

4. Scale and apply
 - helps stabilize gradient
 - turns raw similarity scores into probabilites
 scores = [0.6, 0.3, 0.1]  # sum = 1


6. Sum: Context-aware vector
new_pivot_representation =
  word1 * 0.6 +
  word2 * 0.3 +
  word3 * 0.1

output[i] = new_pivot_representation

7. Repeat for the rest of the words and return this enriched Output sequence!


Another Way to think about it is like this:
1. Column (give us the attention scores)
the (focus) cat sat
the  dot(word1, word1)
cat  ...
sat  ...

2. Then we go into the rows
output[word1] = the*(dot(word1, word1))  + cat * dot(word1, word2) + sat * dot(word1, word1)
"""


# Pseudo Code for self-attention
def self_attention(input_sequence):
  output = np.zeros(shape=input_sequence.shape)
  # iterate over the input sequence
  for i , pivot_vector in enumerate(input_sequence):
    scores = np.zeros(shape=(len(input_sequence),))
    for j, vector in enumerate(input_sequence):
      # Compute attention score/dot product
      scores[j] = np.dot(pivot_vector, vector.T)
    # Normalize and apply a softmax
    scores /= np.sqrt(input_sequence.shape[1])
    scores = softmax(scores)
    new_pivot_representation = np.zeros(shape=pivot_vector.shape)
    for j, vector in enumerate(input_sequence):
      # Take the sum of all tokens weighted by the attention scores.
      new_pivot_representation += vector * scores[j]
    output[i] = new_pivot_representation
  return output


In [2]:
import tensorflow as tf
from keras.layers import MultiHeadAttention

# Generalized Self-attention
"""
                C                      A        B
output = sum(inputs * parwise_scores(inputs, inpupts))

Lets put that in human terms: For each token A, compute how much
it is related to every token in B. Use these scores to weight a sum of
tokens from C.

A, B, and C dont have to be the same inputs.
They are norammly callend query, keys and values

outputs = sum(values * pairwise_scores(query, keys))

In machine translation:
query = target sequence
keys, values are the source sequence

Sequence classification:
query, keys, value = source sequence
"""

# Multi-head attention
"""
The output sapce is separated into manu subspaces that are learned independently.
Then Concatenated back together.

Query Key Value

Query -> Dense Layer --\>
Key -> Dense Layer   --->   Attention Mechanism
Value -> Dense Layer --/>

Q=X WQ

K=X WK

V=X WV

If there would be not Dense layers, then each attention head would do the same!
This enables each head to store a different representation of the same input.
"""
inputs = tf.expand_dims(np.zeros(shape=(100, 100)), axis=-1)
num_heads = 4
embed_dim = 256
mha_layer = MultiHeadAttention(num_heads=num_heads, key_dim=embed_dim)
outputs = mha_layer(inputs, inputs, inputs)


In [3]:
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

# Transformer encoder (used for Text classification, context-awareness, etc.)
class TransformerEncoder(layers.Layer):
  def __init__(self, embed_dim, dense_dim, num_heads, **kwargs):
    super().__init__(**kwargs)
    self.embed_dim = embed_dim # Size of the input token
    self.dense_dim = dense_dim # Size of inner dense layer
    self.num_heads = num_heads
    self.attention = layers.MultiHeadAttention(
        num_heads=num_heads,
        key_dim=embed_dim
        )
    self.dense_proj = keras.Sequential(
        [layers.Dense(dense_dim, activation="relu"),layers.Dense(embed_dim),]
    )
    self.layernorm_1 = layers.LayerNormalization()
    self.layernorm_2 = layers.LayerNormalization()

  def call(self, inputs, mask=None):
    """
    Masked positions are set to very negative values before the softmax,
    so their attention weights become near-zero.

    So that the model is not attending/paying attention to it.

    Example: Padding tokens, etc.
    """

    if mask is not None:
    # The embedding Layer  generates a 2D but attention layer epects to be 3D/4D
      mask = mask[:, tf.newaxis, :]
    attention_output = self.attention(
        inputs, inputs, attention_mask=mask)
    proj_input = self.layernorm_1(inputs + attention_output)
    proj_output = self.dense_proj(proj_input)
    return self.layernorm_2(proj_input + proj_output)

  def get_config(self):
    config = super().get_config()
    config.update({
      "embed_dim": self.embed_dim,
      "num_heads": self.num_heads,
      "dense_dim": self.dense_dim,
    })
    return config


In [4]:
# Load data
!curl -O https://ai.stanford.edu/~amaas/data/sentiment/aclImdb_v1.tar.gz
!tar -xf aclImdb_v1.tar.gz

  % Total    % Received % Xferd  Average Speed   Time    Time     Time  Current
                                 Dload  Upload   Total   Spent    Left  Speed
100 80.2M  100 80.2M    0     0  4783k      0  0:00:17  0:00:17 --:--:-- 6726k


In [5]:
!rm -r aclImdb/train/unsup

In [6]:

import os, pathlib, shutil, random
from tensorflow import keras

# Extract data
base_dir = pathlib.Path("aclImdb")
val_dir = base_dir / "val"
train_dir = base_dir / "train"

for category in ("neg", "pos"):
  os.makedirs(val_dir / category)
  files = os.listdir(train_dir / category)
  random.Random(1337).shuffle(files)
  num_val_samples = int(0.2 * len(files))
  val_files = files[-num_val_samples:]
  for fname in val_files:
    shutil.move(train_dir / category / fname,
                val_dir / category / fname)

# Create sets
batch_size = 32

train_ds = keras.utils.text_dataset_from_directory(
    "aclImdb/train", batch_size=batch_size)
val_ds = keras.utils.text_dataset_from_directory(
    "aclImdb/val", batch_size=batch_size)
test_ds = keras.utils.text_dataset_from_directory(
    "aclImdb/test", batch_size=batch_size)

Found 20000 files belonging to 2 classes.
Found 5000 files belonging to 2 classes.
Found 25000 files belonging to 2 classes.


In [7]:
max_length = 600
max_tokens = 20000
text_vectorization = layers.TextVectorization(
    max_tokens=max_tokens,
    output_mode="int",
    output_sequence_length = max_length,
)

text_only_train_ds = train_ds.map(lambda x, y: x)
text_vectorization.adapt(text_only_train_ds)

int_train_ds = train_ds.map(
    lambda x, y: (text_vectorization(x), y),
    num_parallel_calls=4)
int_val_ds = val_ds.map(
    lambda x, y: (text_vectorization(x), y),
    num_parallel_calls=4)
int_test_ds = test_ds.map(
    lambda x, y: (text_vectorization(x), y),
    num_parallel_calls=4)

In [8]:
vocab_size = 20000
embed_dim = 256
num_heads = 2
dense_dim = 32

inputs = keras.Input(shape=(None,), dtype="int64")
x = layers.Embedding(vocab_size, embed_dim)(inputs)
x = TransformerEncoder(embed_dim, dense_dim, num_heads)(x)
# Reduce the full sequence to a single vector for classifciation
x = layers.GlobalMaxPooling1D()(x)
outputs = layers.Dense(1, activation="sigmoid")(x)
model = keras.Model(inputs, outputs)
model.compile(optimizer="rmsprop",
              loss="binary_crossentropy",
              metrics=["accuracy"])

model.summary()

Model: "functional_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer (InputLayer)        │ (None, None)           │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ embedding (Embedding)           │ (None, None, 256)      │     5,120,000 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ transformer_encoder             │ (None, None, 256)      │       543,776 │
│ (TransformerEncoder)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_max_pooling1d            │ (None, 256)            │             0 │
│ (GlobalMaxPooling1D)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 1)              │           257 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 5,664,033 (21.61 MB)

 Trainable params: 5,664,033 (21.61 MB)

 Non-trainable params: 0 (0.00 B)

In [11]:
callbacks = [
  keras.callbacks.ModelCheckpoint("transformer_encoder.keras",
  save_best_only=True)
]
model.fit(int_train_ds,
          validation_data=int_val_ds,
          epochs=10,callbacks=callbacks)

Epoch 1/10
625/625 ━━━━━━━━━━━━━━━━━━━━ 68s 99ms/step - accuracy: 0.8600 - loss: 0.3284 - val_accuracy: 0.8648 - val_loss: 0.3157
Epoch 2/10
625/625 ━━━━━━━━━━━━━━━━━━━━ 76s 97ms/step - accuracy: 0.8972 - loss: 0.2536 - val_accuracy: 0.8860 - val_loss: 0.2892
Epoch 3/10
625/625 ━━━━━━━━━━━━━━━━━━━━ 85s 102ms/step - accuracy: 0.9305 - loss: 0.1859 - val_accuracy: 0.8906 - val_loss: 0.2877
Epoch 4/10
625/625 ━━━━━━━━━━━━━━━━━━━━ 82s 101ms/step - accuracy: 0.9582 - loss: 0.1219 - val_accuracy: 0.8812 - val_loss: 0.3607
Epoch 5/10
625/625 ━━━━━━━━━━━━━━━━━━━━ 62s 100ms/step - accuracy: 0.9792 - loss: 0.0708 - val_accuracy: 0.8848 - val_loss: 0.3484
Epoch 6/10
625/625 ━━━━━━━━━━━━━━━━━━━━ 77s 92ms/step - accuracy: 0.9911 - loss: 0.0356 - val_accuracy: 0.8836 - val_loss: 0.3856
Epoch 7/10
625/625 ━━━━━━━━━━━━━━━━━━━━ 56s 89ms/step - accuracy: 0.9958 - loss: 0.0189 - val_accuracy: 0.8834 - val_loss: 0.4180
Epoch 8/10
625/625 ━━━━━━━━━━━━━━━━━━━━ 80s 86ms/step - accuracy: 0.9985 - loss: 0.0058

In [13]:
model = keras.models.load_model("transformer_encoder.keras",
                                custom_objects={
                                    "TransformerEncoder": TransformerEncoder})

print(f"Test acc: {model.evaluate(int_test_ds)[1]:.3f}")

782/782 ━━━━━━━━━━━━━━━━━━━━ 15s 18ms/step - accuracy: 0.8796 - loss: 0.3118
Test acc: 0.878


| Model              | Word Order | Awareness Context | Awareness (Cross-Words Interactions) |
|--------------------|------------|-------------------|--------------------------------------|
| Bag-of-unigrams    | No         | No                | No                                   |
| Bag-of-bigrams     | Very limited | No              | No                                   |
| RNN                | Yes        | No                | No                                   |
| Self-attention     | No         | Yes               | Yes                                  |
| Transformer        | Yes        | Yes               | Yes                                  |
